In [1]:
import Gmsh: gmsh
using Gridap, GridapGmsh

In [2]:
function create_cube_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_cube")

    # Use OCC kernel for 3D volumes
    box = gmsh.model.occ.addBox(0.0, 0.0, 0.0, 1.0, 1.0, 1.0)
    gmsh.model.occ.synchronize()

    # Get volume and all boundary surfaces
    vols = gmsh.model.getEntities(3)   # list of (dim, tag)
    surfs = gmsh.model.getEntities(2)

    vol_tags  = [t for (d,t) in vols]
    surf_tags = [t for (d,t) in surfs]

    # Physical groups
    domain_tag   = gmsh.model.addPhysicalGroup(3, vol_tags)
    boundary_tag = gmsh.model.addPhysicalGroup(2, surf_tags)

    gmsh.model.setPhysicalName(3, domain_tag, "domain")
    gmsh.model.setPhysicalName(2, boundary_tag, "boundary")

    # Mesh size control: set characteristic length on all points
    pts = gmsh.model.getEntities(0)
    for (d,p) in pts
        gmsh.model.mesh.setSize([(d,p)], h)
    end

    gmsh.model.mesh.generate(3)

    model = GmshDiscreteModel(gmsh)
    gmsh.finalize()
    return model
end

create_cube_model (generic function with 1 method)

In [3]:
h = 0.04; # target mesh size
model = create_cube_model(h) # fix above function above using the tutorial
order = 1
reffe = ReferenceFE(lagrangian, Float64, order)
V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V, 0.0)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 5 (Line)
Info    : [ 50%] Meshing curve 6 (Line)
Info    : [ 60%] Meshing curve 7 (Line)
Info    : [ 60%] Meshing curve 8 (Line)
Info    : [ 70%] Meshing curve 9 (Line)
Info    : [ 80%] Meshing curve 10 (Line)
Info    : [ 90%] Meshing curve 11 (Line)
Info    : [100%] Meshing curve 12 (Line)
Info    : Done meshing 1D (Wall 0.000597414s, CPU 0.000548s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 2 (Plane, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 3 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing surface 4 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 5 (Plane, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 6 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0701861s

TrialFESpace()

In [12]:
Ω = Triangulation(model)
dΩ = Measure(Ω, 3 * order)

GenericMeasure()

In [13]:
n=6
Δt, T = 2.0^(-n), 2*π/3 # time step, final time
num_steps = Int(round(T / Δt))

134

In [14]:
f(t, x) = 0
f(t) = x -> f(t,x)

f (generic function with 2 methods)

In [15]:
u_exact(t, x) = 0.3*sin(π*x[1])* sin(2*π*x[2])*sin(π*x[3]) * cos(π*sqrt(6)*t) +0.5*sin(2*π*x[1])* sin(2*π*x[2])*sin(3* π*x[3]) * cos(π*sqrt(17)*t) + 0.7 *sin(4*π*x[1])* sin(3*π*x[2])*sin(2*π* x[3]) * cos(π*sqrt(29)*t)

# Initial data (for the PDE)
u₀ = x -> 0.3*sin(π*x[1])* sin(2*π*x[2])*sin(π*x[3]) +0.5*sin(2*π*x[1])* sin(2*π*x[2])*sin(3* π*x[3]) + 0.7 *sin(4*π*x[1])* sin(3*π*x[2])*sin(2*π* x[3])
# u_t(0)=0
u₁ = x -> u_exact(Δt, x) 

#28 (generic function with 1 method)

In [16]:
m(u, v) = ∫(u*v)dΩ
k(u, v) = ∫(∇(u) ⋅ ∇(v))dΩ

M = assemble_matrix(m, U, V)
K = assemble_matrix(k, U, V)

# LHS matrix for (13.5):
A = (1/Δt^2)*M + (1/4)*K

9618×9618 SparseArrays.SparseMatrixCSC{Float64, Int64} with 135570 stored entries:
⎡⠿⢇⠀⠀⠀⠀⣤⣤⣤⣤⣄⣤⣤⣤⣤⣄⣤⣤⣆⣤⣦⣦⣴⣬⣥⣬⣲⣤⣦⣄⣢⣠⣶⣴⣨⣦⣔⣤⣴⣔⎤
⎢⠀⠀⠑⢄⠀⢸⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠀⠀⣀⣀⡑⢌⢻⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠀⣿⣿⣿⣿⣶⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠀⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠀⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠀⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠀⢿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠀⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠈⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠨⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⡐⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⡁⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠘⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠈⢿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠈⣺⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⢘⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠢⣾⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎢⠐⣽⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎥
⎣⢐⢿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⣿⎦

In [17]:
uh0 = interpolate_everywhere(u₀, U)   # U^0
uh1 = interpolate_everywhere(u₁, U)   # U^1a

SingleFieldFEFunction():
 num_cells: 71592
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 11739564403492194914

In [18]:
if !isdir("tmp") mkdir("tmp") end

createpvd("wave") do pvd
    # spara initiala fält
    pvd[0.0] = createvtk(Ω, "tmp/wave4_0.vtu",
                         cellfields=["uh" => uh0, "e" => x -> 0.0])
    pvd[Δt]  = createvtk(Ω, "tmp/wave4_1.vtu",
                         cellfields=["uh" => uh1, "e" => (x -> u_exact(Δt,x)) - uh1])

    # jobba på dof-vektorer (enklast för att bygga RHS med M,K)
    uvec_prev = get_free_dof_values(uh0)  # U^{n-1}
    uvec_curr = get_free_dof_values(uh1)  # U^{n}

    e = nothing
    el2 = nothing

    # Vi kommer räkna fram U^{n+1} från n=1,...,num_steps-1
    for n in 1:(num_steps-1)
        t_n = n*Δt

        # Lastvektor F^n = ∫ f(t_n)*v dΩ
        b(v) = ∫( f(t_n)*v )dΩ
        Fn = assemble_vector(b, V)

        # RHS från (13.5) i matrisform:
        rhs = Fn +
              ((2/Δt^2)*M - (1/2)*K) * uvec_curr +
              (-(1/Δt^2)*M - (1/4)*K) * uvec_prev

        # lös för nästa steg
        uvec_next = A \ rhs
        uh_next = FEFunction(U, uvec_next)

        # fel
        t_next = (n+1)*Δt
        uex = x -> u_exact(t_next, x)
        e = uex - uh_next

        # spara
        vtufilename = "tmp/wave4_$(n+1).vtu"
        pvd[t_next] = createvtk(Ω, vtufilename, cellfields=["uh" => uh_next, "e" => e])

        # uppdatera
        uvec_prev = uvec_curr
        uvec_curr = uvec_next
    end

    # sista felet (från sista iterationen)
    el2 = sqrt(sum( ∫(e*e)*dΩ ))
    println("L2 error norm: $el2")
end

L2 error norm: 0.2075319746901554


136-element Vector{String}:
 "wave.pvd"
 "tmp/wave4_0.vtu"
 "tmp/wave4_1.vtu"
 "tmp/wave4_2.vtu"
 "tmp/wave4_3.vtu"
 "tmp/wave4_4.vtu"
 "tmp/wave4_5.vtu"
 "tmp/wave4_6.vtu"
 "tmp/wave4_7.vtu"
 "tmp/wave4_8.vtu"
 "tmp/wave4_9.vtu"
 "tmp/wave4_10.vtu"
 "tmp/wave4_11.vtu"
 ⋮
 "tmp/wave4_123.vtu"
 "tmp/wave4_124.vtu"
 "tmp/wave4_125.vtu"
 "tmp/wave4_126.vtu"
 "tmp/wave4_127.vtu"
 "tmp/wave4_128.vtu"
 "tmp/wave4_129.vtu"
 "tmp/wave4_130.vtu"
 "tmp/wave4_131.vtu"
 "tmp/wave4_132.vtu"
 "tmp/wave4_133.vtu"
 "tmp/wave4_134.vtu"